In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import rand, when

In [0]:
spark = (SparkSession.builder
         .appName("partitioning-and-repartitioning")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [0]:
# Create some sample data frames
# A large data frame with 1 million rows
large_df = (spark.range(0, 1000000)
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("country_code", 
                        when((rand() * 4).cast("int") == 0, "US")
                        .when((rand() * 4).cast("int") == 1, "CN")
                        .when((rand() * 4).cast("int") == 2, "IN")
                        .when((rand() * 4).cast("int") == 3, "BR")))
large_df.show(5)

In [0]:
num_partitions = large_df.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = large_df.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

In [0]:
df_hash = large_df.repartition(10, "id")

In [0]:
num_partitions = df_hash.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = df_hash.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

In [0]:
df_range = large_df.repartitionByRange(10, "id")

In [0]:
num_partitions = df_range.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = df_range.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

In [0]:
df_coalesce = df_range.coalesce(4)

In [0]:
num_partitions = df_coalesce.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = df_coalesce.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

In [0]:
(large_df.write 
    .format("parquet")
    .partitionBy("id")
    .mode("overwrite")
    .save("../data/tmp/partitioned_output"))

In [0]:
df_read = (spark.read
           .format("parquet")
           .load("../data/tmp/partitioned_output"))

df_read.show(5)

In [0]:
spark.stop()